In [ ]:
import json
import urllib.error
import urllib.request

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scipy.stats as stats
import itertools

import altair as alt
from sklearn.metrics import precision_recall_curve, auc

In [ ]:
input_directory = '/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/vampseq_data_for_qc'

files = [
    "./Data/vampseq_data_for_qc/" + f
    for f in os.listdir(input_directory)
    if f.endswith(".csv")
]

file_dict = {}
for file in files:
    file_dict_key = file.split('_scores')[0].split('20260410_')[1]

    file_dict[file_dict_key] = file
print(file_dict)
alt.data_transformers.disable_max_rows()

In [ ]:
# Scatter plot helper function

def corr_scatter(df, rep1, rep2, gene):

    rep1_max = df.max(axis = 0)[rep1]
    rep1_min = df.min(axis = 0)[rep1]
    rep2_max = df.max(axis = 0)[rep2]
    rep2_min = df.min(axis = 0)[rep2]

    x_max = rep1_max * 1.05
    y_max = rep2_max * 1.05
    y_min = rep2_min * 1.05

    df = df.dropna(subset = [rep1, rep2]).copy()
    
    scatter = alt.Chart(df).mark_circle().encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)
                  ),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  )
    )

    corr,_=stats.pearsonr(df[rep1], df[rep2])

    r_text = alt.Chart(pd.DataFrame({
        rep1: [x_max * 0.95],
        rep2: [0],
        'text': [f'r = {corr:.3f}']
    })).mark_text(
        align='right',
        baseline='bottom',
        fontSize=18,
        fontWeight='bold',
        color='black'
    ).encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        text='text:N'
    )

    scatter = (scatter + r_text + scatter.transform_regression(rep1, rep2).mark_line(color = 'red')).properties(title = gene).resolve_scale(x = 'shared', y = 'shared')

    rep_test = f'{rep1} vs. {rep2}'
    return scatter, gene, rep_test, corr

# G6PD correlation

In [ ]:
g6pd_path = file_dict['G6PD']
g6pd_df = pd.read_csv(g6pd_path)

g6pd_df = g6pd_df.rename(columns = {'rep1_score': 'R1', 'rep3_score': 'R2'})

g6pd_df = g6pd_df[~g6pd_df['standard_error'].isin([0, 'NA'])].copy()

g6pd_df = g6pd_df.dropna(how = 'any')
g6pd_df.head()

In [ ]:
g6pd_scatter, gene_name, rep_test, correlation = corr_scatter(g6pd_df, 'R1', 'R2', 'G6PD')

g6pd_scatter.display()

In [ ]:
corr_df_tuple = [(gene_name, rep_test, correlation)]
corr_df = pd.DataFrame(corr_df_tuple, columns=['gene', 'replicates', 'pearson_r'])

corr_df.head()

# FIX Correlation

In [ ]:
indv_corr_dfs = []
for key in file_dict.keys():
    if 'FIX' in key and 'Strep' not in key:
        print(f'----{key}----')
        fix_path = file_dict[key]

        fix_df = pd.read_csv(fix_path)

        fix_df = fix_df[~fix_df['N_replicates'].isin([9, 0, 1])].copy()

        tiles = ['Tile1', 'Tile2', 'Tile3']
        fix_dfs = {}

        for tile in tiles:
            if len(fix_df.columns.tolist()) == 14:
                df = fix_df[['accession', 'hgvs_pro', 'score', 'SE_score', 'N_replicates', f'Score_{tile}_BioReplicate1', f'Score_{tile}_BioReplicate2', f'Score_{tile}_BioReplicate3']].copy()
                df = df.rename(columns = {f'Score_{tile}_BioReplicate1': "R1", f'Score_{tile}_BioReplicate2': "R2", f'Score_{tile}_BioReplicate3': "R3"})
                df.dropna(subset = ['R1', 'R2', 'R3'])
            else:
                df = fix_df[['accession', 'hgvs_pro', 'score', 'SE_score', 'N_replicates', f'Score_{tile}_BioReplicate1', f'Score_{tile}_BioReplicate2']].copy()
                df = df.rename(columns = {f'Score_{tile}_BioReplicate1': "R1", f'Score_{tile}_BioReplicate2': "R2"})
                df.dropna(subset = ['R1', 'R2'])

            df_key = key + "_" + tile
            fix_dfs[df_key] = df
        
        plots = []

        df_tuples = []
        for tile in fix_dfs.keys():
            df = fix_dfs[tile]

            if 'R3' not in df.columns.tolist():
                score_cols = ['R1', 'R2']
            else:
                score_cols = ['R1', 'R2', 'R3']
            r_combos = list(itertools.combinations(score_cols, 2))
            
            tile_plots = []
            for combo in r_combos:
                rep1, rep2 = combo

                tile_scatter, fix_tile, rep_test, r_corr = corr_scatter(df,rep1,rep2, f'F9 {tile}')

                df_tuples.append((fix_tile, rep_test, r_corr))
                tile_plots.append(tile_scatter)
            
            final_tile_plot = alt.hconcat(*tile_plots)
            plots.append(final_tile_plot)
        fix_plot = alt.vconcat(*plots).display()
        fix_corr_df = pd.DataFrame(df_tuples, columns = ['gene', 'replicates', 'pearson_r'])
        indv_corr_dfs.append(fix_corr_df)
        
fix_corr_df = pd.concat(indv_corr_dfs)


In [ ]:
fix_corr_df.head(n = 30)

# TSC2 Correlation

In [ ]:
tsc2_lib1_path = file_dict['TSC2_lib1']
tsc2_lib2_path = file_dict['TSC2_lib2']

lib1_df = pd.read_csv(tsc2_lib1_path)
lib1_df = lib1_df.rename(columns = {'abundance_1': "R1", "abundance_2": "R2"})

lib2_df = pd.read_csv(tsc2_lib2_path)
lib2_df = lib2_df.rename(columns = {'abundance_1': "R1", "abundance_2": "R2", "abundance_3": "R3"})

tsc2_df = pd.concat([lib1_df, lib2_df])
tsc2_df['hgvs_pro'] = tsc2_df['variant'].transform(lambda x: x.split(':')[1])

tsc2_plots = []
tsc2_corr_tuples = []

tsc2_lib1_plot, tsc2_tile1, tsc2_tile1_rep, tsc2_tile1_r_corr = corr_scatter(lib1_df, 'R1', 'R2', 'TSC2 Tile 1')

tsc2_corr_tuples.append((tsc2_tile1, tsc2_tile1_rep, tsc2_tile1_r_corr))
tsc2_plots.append(tsc2_lib1_plot)

In [ ]:
lib2_combos = list(itertools.combinations(['R1', 'R2', 'R3'], 2))

lib2_plots = []
for combo in lib2_combos:
    rep1, rep2 = combo

    lib_scatter,lib_name, lib_reps, lib_r_corr = corr_scatter(lib2_df,rep1,rep2, f'TSC2 Tile 2')

    tsc2_corr_tuples.append((lib_name, lib_reps, lib_r_corr))

    lib2_plots.append(lib_scatter)

final_lib2_plot = alt.hconcat(*lib2_plots)
tsc2_plots.append(final_lib2_plot)

final_tsc2_plot = alt.vconcat(*tsc2_plots)

final_tsc2_plot.display()

In [ ]:
tsc2_r_corr_df = pd.DataFrame(tsc2_corr_tuples, columns = ['gene', 'replicates', 'pearson_r'])
tsc2_r_corr_df.head()

# Pearson r Heatmap

In [ ]:
corr_df = pd.concat([corr_df, fix_corr_df, tsc2_r_corr_df]).reset_index(drop = True)
corr_df.head(n = 30)

In [ ]:
gene_order = ['G6PD', 'TSC2 Tile 1', 'TSC2 Tile 2', 'F9 Tile1', 'F9 Tile2', 'F9 Tile3']
rep_order = ['R2 vs. R3', 'R1 vs. R3', 'R1 vs. R2']

base = alt.Chart(corr_df).encode(
    x=alt.X('gene:N', sort=gene_order),
    y=alt.Y('replicates:N', sort=rep_order)
)

heatmap = base.mark_rect().encode(
    color=alt.Color('pearson_r:Q', scale=alt.Scale(domain=[0.5, 1.0]))
)

text_color = (
    alt.when(alt.datum.pearson_r > 0.5)
    .then(alt.value('white'))
    .otherwise(alt.value('black'))
)

text = base.mark_text(baseline='middle', fontSize=14, font='Arial').encode(
    text=alt.Text('pearson_r:Q', format='0.3f'),
    color=text_color,
).transform_filter('isValid(datum.pearson_r)')

(heatmap + text).properties(
    width=alt.Step(45),
    height=alt.Step(30)
).configure_axis(
    labelFont='Arial',
    titleFont='Arial',
).configure_legend(
    labelFont='Arial',
    titleFont='Arial',
).configure_view(
    stroke = None
).configure_axis(
    grid = False
). display()

# Gene Cartoon Helper Functions

In [ ]:
_SPECIES_MAP = {
    "human":       "homo_sapiens",
    "mouse":       "mus_musculus",
    "rat":         "rattus_norvegicus",
    "zebrafish":   "danio_rerio",
    "fly":         "drosophila_melanogaster",
    "worm":        "caenorhabditis_elegans",
    "yeast":       "saccharomyces_cerevisiae",
}


def _fetch_gene_data(gene_symbol: str, species: str, assembly: str) -> dict:
    """Fetch raw gene data from the Ensembl REST API (internal helper)."""
    ens_species = _SPECIES_MAP.get(species.lower(), species.lower())
    base_url = (
        "https://grch37.rest.ensembl.org"
        if assembly.upper() == "GRCH37"
        else "https://rest.ensembl.org"
    )
    url = (
        f"{base_url}/lookup/symbol/{ens_species}/{gene_symbol}"
        "?expand=1&content-type=application/json"
    )
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            return json.loads(resp.read())
    except urllib.error.HTTPError as e:
        raise ValueError(
            f"Ensembl REST API returned HTTP {e.code} for gene '{gene_symbol}'. "
            "Check that the gene symbol and species are correct."
        ) from e
    except urllib.error.URLError as e:
        raise ConnectionError(
            f"Could not reach Ensembl REST API: {e.reason}. "
            "Check your internet connection."
        ) from e


def _select_transcript(data: dict, gene_symbol: str, transcript_id: str | None):
    """Pick a transcript dict from raw Ensembl gene data (internal helper)."""
    transcripts = data.get("Transcript", [])
    if not transcripts:
        raise ValueError(f"No transcripts found for gene '{gene_symbol}'.")
    if transcript_id is not None:
        tx = next((t for t in transcripts if t["id"] == transcript_id), None)
        if tx is None:
            available = [t["id"] for t in transcripts]
            raise ValueError(
                f"Transcript '{transcript_id}' not found for '{gene_symbol}'. "
                f"Available: {available}"
            )
        return tx
    canonical = [t for t in transcripts if t.get("is_canonical") == 1]
    return canonical[0] if canonical else max(
        transcripts,
        key=lambda t: (t.get("Translation") is not None, t.get("length", 0)),
    )

In [ ]:
def get_canonical_transcript(
    gene_symbol: str,
    species: str = "human",
    assembly: str = "GRCh38",
) -> dict:
    """Return a summary dict for the canonical transcript of a gene.

    Fetches from the Ensembl REST API and returns a dict with keys:
      transcript_id, biotype, n_exons, is_canonical, strand.

    Useful for inspecting the auto-selected transcript before calling
    fetch_exon_coords().
    """
    data = _fetch_gene_data(gene_symbol, species, assembly)
    tx = _select_transcript(data, gene_symbol, transcript_id=None)
    return {
        "transcript_id": tx["id"],
        "biotype":       tx.get("biotype", "unknown"),
        "n_exons":       len(tx.get("Exon", [])),
        "is_canonical":  bool(tx.get("is_canonical") == 1),
        "strand":        "minus" if data["strand"] == -1 else "plus",
        "_raw_data":     data,  # passed through to avoid a second API call
    }


def fetch_exon_coords(
    gene_symbol: str,
    transcript_id: str | None = None,
    species: str = "human",
    assembly: str = "GRCh38",
    _raw_data: dict | None = None,
) -> tuple[pd.DataFrame, None, pd.DataFrame]:
    """Fetch exon coordinates for a gene from the Ensembl REST API.

    Returns (exon_df, None, meta_df) matching the load_cartoon() output
    format, so the result can be passed directly to make_library_cartoon().

    Args:
        gene_symbol:   HGNC gene symbol (e.g. "BRCA1").
        transcript_id: Ensembl transcript ID to use. If None, the canonical
                       transcript is selected automatically.
        species:       Common name (e.g. "human") or Ensembl species name.
        assembly:      Genome assembly — "GRCh38" (default) or "GRCh37".
    """
    data = _raw_data or _fetch_gene_data(gene_symbol, species, assembly)
    tx = _select_transcript(data, gene_symbol, transcript_id)
    strand = "minus" if data["strand"] == -1 else "plus"

    # Sort in transcription order (5'->3') so X1 is always the first exon.
    exons = sorted(tx.get("Exon", []), key=lambda e: e["start"], reverse=(strand == "minus"))
    exon_df = pd.DataFrame([
        {"exon": f"X{i + 1}", "start": e["start"], "end": e["end"]}
        for i, e in enumerate(exons)
    ])

    translation = tx.get("Translation")
    if translation is not None:
        atg_pos  = translation["end"]   if strand == "minus" else translation["start"]
        stop_pos = translation["start"] if strand == "minus" else translation["end"]
    else:
        atg_pos  = int(exon_df["start"].min())
        stop_pos = int(exon_df["end"].max())

    meta_df = pd.DataFrame([
        {"type": "strand", "info": strand},
        {"type": "atg",    "info": atg_pos},
        {"type": "stop",   "info": stop_pos},
    ])

    print(
        f"  Fetched {len(exon_df)} exons for {gene_symbol} "
        f"({tx['id']}, {strand}-strand, {assembly})"
    )
    return exon_df, None, meta_df

In [ ]:
def fetch_protein_length(
    gene_symbol: str,
    transcript_id: str | None = None,
    species: str = "human",
    assembly: str = "GRCh38",
) -> int:
    """Return the protein length (in amino acids) for the canonical transcript.

    Args:
        gene_symbol:   HGNC gene symbol (e.g. "F9").
        transcript_id: Ensembl transcript ID to use. If None, the canonical
                       transcript is selected automatically.
        species:       Common name or Ensembl species name (default "human").
        assembly:      "GRCh38" (default) or "GRCh37".
    """
    data = _fetch_gene_data(gene_symbol, species, assembly)
    tx   = _select_transcript(data, gene_symbol, transcript_id)
    translation = tx.get("Translation")
    if translation is None:
        raise ValueError(
            f"Transcript {tx['id']} for '{gene_symbol}' has no Translation "
            "(non-coding transcript?). Specify a protein-coding transcript_id."
        )
    length = translation["length"]
    print(f"  {gene_symbol} — {tx['id']}: {length} aa")
    return length


def get_aa_range_from_scores(
    scores_df: pd.DataFrame,
    hgvs_col: str = "hgvs_pro",
    gap_threshold: int = 1,
) -> list[tuple[int, int]]:
    """Return contiguous AA intervals covered by a scores file.

    Extracts every unique residue number from the HGVS protein column,
    sorts them, and groups into runs where consecutive positions differ by
    at most gap_threshold.  Each run becomes one (start, end) interval.

    A gap_threshold of 1 (default) means any missing amino acid position
    splits the coverage into separate intervals.

    Args:
        scores_df:     DataFrame with a column of HGVS protein variant strings.
        hgvs_col:      Name of the HGVS protein column (default 'hgvs_pro').
        gap_threshold: Maximum allowed gap (in AA positions) within one interval.
    """
    positions = (
        scores_df[hgvs_col]
        .dropna()
        .str.extractall(r"(\d+)")[0]
        .astype(int)
        .drop_duplicates()
        .sort_values()
        .to_numpy()
    )
    if len(positions) == 0:
        raise ValueError(f"No residue numbers found in column '{hgvs_col}'.")

    intervals: list[tuple[int, int]] = []
    start = prev = int(positions[0])
    for pos in positions[1:]:
        pos = int(pos)
        if pos - prev > gap_threshold:
            intervals.append((start, prev))
            start = pos
        prev = pos
    intervals.append((start, prev))
    return intervals

In [ ]:
def load_domain_data(path: str, gene_symbol: str) -> pd.DataFrame:
    """Load domain annotations for one gene from the shared domains Excel file.

    Expects a sheet named after the gene (e.g. "F9") with columns:
        domain  – domain name
        start   – first amino acid position (inclusive)
        end     – last amino acid position (inclusive)

    Returns a DataFrame with those three columns, or an empty DataFrame if
    the gene has no sheet in the file.

    Args:
        path:        Path to the domains Excel workbook.
        gene_symbol: Gene name matching the sheet name (e.g. "F9").
    """
    xl = pd.ExcelFile(path)
    if gene_symbol not in xl.sheet_names:
        print(f"  No domain sheet found for '{gene_symbol}' — skipping domains.")
        return pd.DataFrame(columns=["domain", "start", "end"])
    df = xl.parse(gene_symbol)[["domain", "start", "end"]]
    print(f"  Loaded {len(df)} domain(s) for {gene_symbol}.")
    return df

In [ ]:
_DOMAIN_COLORS = [
    "#B9DBF4", "#C8DBC8", "#F6BF93", "#D5D0F2",
    "#018571", "#D35400", "#2980B9", "#C0392B",
]


def make_library_cartoon(
    gene: str,
    protein_length: int,
    covered_intervals: list[tuple[int, int]],
    n_variants: int,
    n_tiles: int,
    domain_df: pd.DataFrame | None = None,
    fig_width: float = 10,
    fig_height: float = 1.2,
    gene_color: str = "#a9a9a9",
    hatch: str = "//////",
) -> plt.Figure:
    """Draw a gene rectangle with optional domain coloring and coverage hatching.

    Layout (top to bottom):
      - Gene rectangle spanning 1 → protein_length (gene_color).
      - Domain rectangles overlaid on the gene rect, each in a distinct color.
      - Hatch overlay on every covered interval (data coverage).
      - AA position ticks and labels (1, every 100, and protein_length).
      - Domain name labels (if domain_df provided).
      - Gene name, variant count, and tile count to the right of the cartoon.

    Args:
        gene:              Gene symbol, shown in the right-side label.
        protein_length:    Total protein length in amino acids.
        covered_intervals: List of (aa_start, aa_end) tuples from
                           get_aa_range_from_scores().
        n_variants:        Number of variants in the scores data.
        n_tiles:           Number of library tiles for this gene.
        domain_df:         DataFrame with columns [domain, start, end].
                           Pass None to skip domain annotation.
        fig_width:         Figure width in inches.
        fig_height:        Figure height in inches.
        gene_color:        Fill colour for uncovered portions of the gene rect.
        hatch:             Matplotlib hatch pattern for covered regions.
    """
    has_domains = domain_df is not None and not domain_df.empty

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # ── Base gene rectangle ───────────────────────────────────────────────────
    ax.add_patch(mpatches.Rectangle(
        (1, 0), protein_length - 1, 1,
        facecolor=gene_color, edgecolor="black", linewidth=0.8,
        zorder=1,
    ))

    # ── Domain rectangles ─────────────────────────────────────────────────────
    if has_domains:
        for i, (_, row) in enumerate(domain_df.iterrows()):
            color = _DOMAIN_COLORS[i % len(_DOMAIN_COLORS)]
            ax.add_patch(mpatches.Rectangle(
                (row["start"], 0), row["end"] - row["start"], 1,
                facecolor=color, edgecolor="black", linewidth=0.8,
                zorder=2,
            ))

    # ── Hatch overlay for covered intervals ───────────────────────────────────
    for aa_start, aa_end in covered_intervals:
        ax.add_patch(mpatches.Rectangle(
            (aa_start, 0), aa_end - aa_start, 1,
            facecolor="none", edgecolor="black", linewidth=0,
            hatch=hatch, zorder=3,
        ))

    # ── AA position ticks and labels ──────────────────────────────────────────
    hundred_ticks = list(range(100, protein_length, 100))
    if hundred_ticks and protein_length - hundred_ticks[-1] <= 30:
        hundred_ticks = hundred_ticks[:-1]
    tick_positions = sorted(set([1] + hundred_ticks + [protein_length]))
    for pos in tick_positions:
        ax.plot([pos, pos], [0, -0.12], color="black", linewidth=0.8, zorder=4)
        ax.text(pos, -0.20, str(pos), ha="center", va="top", fontsize=10, fontweight="bold", font="arial")

    # ── Domain labels centered below ticks ────────────────────────────────────
    if has_domains:
        for i, (_, row) in enumerate(domain_df.iterrows()):
            center_x = (row["start"] + row["end"]) / 2
            color = _DOMAIN_COLORS[i % len(_DOMAIN_COLORS)]
            ax.text(
                center_x, -0.60, row["domain"],
                ha="center", va="top", fontsize=12, fontweight="bold", font="arial",
                color="black",
            )

    # ── Right-side label: gene / variants / tiles ─────────────────────────────
    y_min = -0.85 if has_domains else -0.45
    y_bar_center = (0.5 - y_min) / (1.2 - y_min)   # gene bar center in axes fraction
    spacing = 0.20                                   # between lines, in axes fraction
    for txt, offset, bold in [
        (gene,                       spacing, True),
        (f"{n_variants:,} Variants",       0, False),
        (f"{n_tiles} Tiles",        -spacing, False),
    ]:
        ax.text(
            1.03, y_bar_center + offset, txt,
            transform=ax.transAxes, clip_on=False,
            ha="left", va="center",
            fontsize=11 if bold else 9,
            fontweight="bold" if bold else "normal",
            font="arial",
        )

    ax.set_xlim(0, protein_length + 1)
    ax.set_ylim(y_min, 1.2)
    ax.axis("off")
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.savefig(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/vampseq/20260410_{gene}_cartoon.svg', format = 'svg')
    return fig

# Gene Cartoon

In [ ]:
# Placeholder tile counts 
tile_counts = pd.DataFrame([
    {"gene": "F9",   "n_tiles": 3},
    {"gene": "G6PD", "n_tiles": 1},
    {"gene": "TSC2", "n_tiles": 2},
])

gene_dfs = {'G6PD': g6pd_df,
            'TSC2': tsc2_df,
            'F9': fix_df
            }

In [ ]:
DOMAINS_PATH = "./Data/vampseq_data_for_qc/20260413_VAMPseq_protein_domains.xlsx"

for i, gene in enumerate(gene_dfs.keys()):
    df = gene_dfs[gene]
    protein_length    = fetch_protein_length(gene)
    covered_intervals = get_aa_range_from_scores(df)
    domain_df         = load_domain_data(DOMAINS_PATH, gene)
    n_variants        = len(df)
    n_tiles           = tile_counts.loc[tile_counts["gene"] == gene, "n_tiles"].iloc[0]

    make_library_cartoon(
        gene, protein_length, covered_intervals,
        n_variants, n_tiles, domain_df,
    )

In [ ]:
fig, ax = plt.subplots(figsize=(5, 0.6))

patch_kw = dict(height=0.6, edgecolor='black')
ax.add_patch(mpatches.Rectangle((0, 0), 1, facecolor='#a9a9a9', linewidth=0.8, **patch_kw))
ax.add_patch(mpatches.Rectangle((0, 0), 1, facecolor='none', hatch='//////', linewidth=0, **patch_kw))
ax.text(1.2, 0.3, 'Data Coverage', va='center', fontsize=10, font='arial')

ax.add_patch(mpatches.Rectangle((5, 0), 1, facecolor="#FFFFFF", linewidth=0.8, **patch_kw))
ax.text(6.2, 0.3, 'No Data', va='center', fontsize=10, font='arial')

ax.set_xlim(0, 10)
ax.set_ylim(-0.1, 0.8)
ax.axis('off')
plt.tight_layout()
plt.show()
fig.savefig('/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/vampseq/20260410_gene_cartoon_legend.svg', format = 'svg')

# Precision-Recall analysis

In [ ]:
def make_pr_curve(
    df: pd.DataFrame,
    score_col: str = "auth_reported_score",
    label_col: str = "clinvar_sig_2025",
    pos_label: str = "Pathogenic",
    neg_label: str = "Benign",
    title: str = "Precision-Recall",
) -> alt.Chart:
    """Build a Precision-Recall curve chart using Altair.

    Rows with labels other than pos_label/neg_label (e.g. 'VUS')
    are excluded. Lower auth_reported_score = more likely abnormal (positive),
    so scores are negated before computing the curve.

    Parameters
    ----------
    df : DataFrame containing score_col and label_col.
    score_col : Column with continuous predictor scores.
    label_col : Column with functional class labels.
    pos_label : Label string treated as the positive class.
    neg_label : Label string treated as the negative class.
    title : Chart title prefix; AUC-PR is appended automatically.

    Returns
    -------
    alt.Chart
    """
    sub = df[df[label_col].isin([pos_label, neg_label])].copy()
    y_true = (sub[label_col] == pos_label).astype(int)
    # Negate: lower score = more abnormal, sklearn expects higher = more positive
    y_score = -sub[score_col]

    precision, recall, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall, precision)
    baseline = y_true.mean()

    curve_df = pd.DataFrame({"Recall": recall, "Precision": precision})

    curve = (
        alt.Chart(curve_df, title=f"{title} (AUC-PR = {pr_auc:.3f})")
        .mark_line(color="orange")
        .encode(
            x=alt.X("Recall:Q", scale=alt.Scale(domain=[0, 1]),
                    axis=alt.Axis(title="Recall", labelFont="Arial", titleFont="Arial")),
            y=alt.Y("Precision:Q", scale=alt.Scale(domain=[baseline * 0.95, 1]),
                    axis=alt.Axis(title="Precision", labelFont="Arial", titleFont="Arial")),
            tooltip=[
                alt.Tooltip("Recall:Q", format=".3f"),
                alt.Tooltip("Precision:Q", format=".3f"),
            ],
        )
        .properties(width=400, height=350)
    )

    baseline_df = pd.DataFrame({"y": [baseline]})
    baseline_rule = (
        alt.Chart(baseline_df)
        .mark_rule(color="gray", strokeDash=[4, 4])
        .encode(y="y:Q")
    )

    final_plot = (curve + baseline_rule).configure_title(font="Arial", fontSize=14).configure_axis(grid = False).configure_view(stroke = None)
    final_plot.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/vampseq/20240414_{gene}_PRcurve.svg')

    return final_plot

In [ ]:
ppj_df = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/20260101_VAMPseqsubset_wDups.xlsx')


In [ ]:
genes = ['F9', 'G6PD', 'TSC2']

gene_ppj_clinvar_dfs = {}

clinvar_mapping = {
    'Likely benign': 'Benign',
    'Benign/Likely benign': 'Benign',
    'Likely pathogenic': 'Pathogenic',
    'Pathogenic/Likely pathogenic': 'Pathogenic',
    'Uncertain significance': "VUS",
    'not provided': 'VUS',
    'Conflicting classifications of pathogenicity': 'VUS',
    'no classification for the single variant': "VUS"
}

for gene in genes:
    gene_df = ppj_df.loc[ppj_df['Gene'] == gene].copy()
    gene_df['clinvar_sig_2025'] = gene_df['clinvar_sig_2025'].map(clinvar_mapping)

    #gene_df = gene_df.dropna(subset = ['clinvar_sig'])
    #gene_df = gene_df[~gene_df['clinvar_sig'].isin(['VUS'])].copy()

    gene_ppj_clinvar_dfs[gene] = gene_df



In [ ]:

for gene in gene_ppj_clinvar_dfs.keys():
    df = gene_ppj_clinvar_dfs[gene]

    print(f'Processing {gene}...')
    print(len(df))
    if len(df) > 10:
        curve = make_pr_curve(df, title=gene)
        curve.display()
    else:
        print(f'{gene} did not have enough ClinVar variants')
